# AF2 spectral — staged global decision
Attach the private core dataset plus exactly three output ZIPs: `af2-spectral-stage1-sequential-output.zip`, `PCG1_seed42_output.zip`, and `WAV1_seed42_output.zip`. This notebook extracts those archives, reconstructs the seven seed-42 result JSONs, computes Stage-1, Stage-2, and global frozen decisions, and never trains or opens test data.


In [ ]:
import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input')
REPO=WORK/'coffee-bean-detection'; OUT=WORK/'af2-spectral-factorization-v1'; REPORTS=OUT/'val_reports'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

EXTRACT=WORK/'af2-spectral-global-input'
if EXTRACT.exists(): shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)
required_zips=('af2-spectral-stage1-sequential-output.zip','PCG1_seed42_output.zip','WAV1_seed42_output.zip')
for name in required_zips:
    matches=sorted(path for path in INPUT.rglob(name) if path.is_file())
    if len(matches)!=1: raise FileNotFoundError(f'Harus ada tepat satu {name}; ditemukan {matches}')
    destination=EXTRACT/Path(name).stem
    destination.mkdir(parents=True)
    with zipfile.ZipFile(matches[0],'r') as handle: handle.extractall(destination)
    print('EXTRACTED:',matches[0],'->',destination)

REPORTS.mkdir(parents=True,exist_ok=True)
arms=('AF2WIN','AF2ORI','AF2POL','AF2SOFT','AF2LUM','PCG1','WAV1')
for arm in arms:
    matches=sorted(path for path in EXTRACT.rglob(f'{arm}_seed42_result.json') if path.is_file())
    if len(matches)!=1: raise FileNotFoundError(f'Harus ada tepat satu hasil {arm}; ditemukan {matches}')
    payload=json.loads(matches[0].read_text(encoding='utf-8'))
    assert payload['arm']==arm and payload['seed']==42 and payload['evaluation_split']=='val' and payload['test_images_accessed'] is False
    shutil.copy2(matches[0],REPORTS/matches[0].name)
    print('RESULT READY:',arm,matches[0])

baseline=sorted(path for path in INPUT.rglob('lfdet_afab_seed42_screening.json') if path.is_file())
if len(baseline)!=1: raise FileNotFoundError(f'Harus ada tepat satu evidence AF2; ditemukan {baseline}')
from coffee_detector.experiments.run_faruq_v3_af2_spectral_decision import run_spectral_decision
stage1=run_spectral_decision(OUT,baseline[0],stage='stage1')
assert stage1['test_opened'] is False and stage1['next']=='AUTHORIZE_STAGE2'
stage2=run_spectral_decision(OUT,baseline[0],stage='stage2')
assert stage2['test_opened'] is False and stage2['next']=='AUTHORIZE_GLOBAL_DECISION'
global_result=run_spectral_decision(OUT,baseline[0],stage='global')
assert global_result['test_opened'] is False
print('=== GLOBAL RESULT ===')
print(json.dumps(global_result,indent=2))
archive=shutil.make_archive('/kaggle/working/af2-spectral-global-decision','zip',OUT)
print('DOWNLOAD SEBELUM STOP SESSION:',archive)
